# LoRA Fine-tuning with SmolLM2 135M using Unsloth

This notebook demonstrates LoRA (Low-Rank Adaptation) fine-tuning of the SmolLM2 135M parameter model using Unsloth. Unlike full fine-tuning, LoRA only updates a small set of adapter weights, making it much more memory-efficient.

## What we'll cover:
- Installing Unsloth and dependencies
- Loading the SmolLM2 135M model with 4-bit quantization
- Configuring LoRA adapters with specific rank and target modules
- Training with LoRA (only adapter weights are updated)
- Testing inference with the fine-tuned model
- Comparing LoRA vs Full Fine-tuning

## About LoRA:
LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning technique that freezes the pre-trained model weights and injects trainable low-rank decomposition matrices into each layer. This dramatically reduces:
- Number of trainable parameters (typically 1-10% of original model)
- GPU memory requirements (can use 4-bit quantization)
- Training time

## Key Differences from Full Fine-tuning:
- Only trains adapter weights, not the full model
- Can use 4-bit quantization for even lower memory usage
- Much faster training
- Smaller saved models (only adapters need to be saved)

In [1]:
# Install Unsloth and dependencies - optimized for Colab
# This version uses pre-built wheels to avoid long compilation times

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-j3n4z9dy/unsloth_0f63f77bb5154d90a98260ffd3c271b8
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-j3n4z9dy/unsloth_0f63f77bb5154d90a98260ffd3c271b8
  Resolved https://github.com/unslothai/unsloth.git to commit 341ce85864d191e4a6b7c447b9167c1faf5e20d3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.5/283.5 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 136.1 MB/s eta 0:00:00

In [2]:
# Import necessary libraries
from unsloth import FastLanguageModel
import torch
from transformers import TrainingArguments
from trl import SFTTrainer
from datasets import load_dataset

# Check if GPU is available
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU Available: True
GPU Name: NVIDIA L4
GPU Memory: 23.80 GB


## Model Configuration for LoRA Fine-tuning

For LoRA fine-tuning, we can use 4-bit quantization to significantly reduce memory usage since we're only training small adapter weights, not the full model.

Key differences from Full Fine-tuning:
- `load_in_4bit`: Set to **True** (we can use quantization with LoRA)
- We will add LoRA adapters with `get_peft_model` instead of setting `full_finetuning=True`

Configuration parameters:
- `max_seq_length`: Maximum sequence length for training (2048 tokens)
- `dtype`: Data type for computation (None means auto-detection)
- `load_in_4bit`: **True** for LoRA (enables 4-bit quantization for memory efficiency)
- `model_name`: The specific model to use from HuggingFace

In [3]:
# Configuration parameters for LoRA fine-tuning
max_seq_length = 2048  # Maximum sequence length
dtype = None  # Auto-detect dtype (will use float16 for efficiency)
load_in_4bit = True  # Use 4-bit quantization (possible with LoRA)

# Model selection - using SmolLM2 135M (same model as Colab 1)
model_name = "unsloth/SmolLM2-135M-Instruct"

print(f"Model: {model_name}")
print(f"Max Sequence Length: {max_seq_length}")
print(f"4-bit Quantization: {load_in_4bit}")
print("LoRA fine-tuning enabled (parameter-efficient)")

Model: unsloth/SmolLM2-135M-Instruct
Max Sequence Length: 2048
4-bit Quantization: True
LoRA fine-tuning enabled (parameter-efficient)


## Load Model and Configure LoRA Adapters

We load the model with 4-bit quantization and then add LoRA adapters using `get_peft_model`.

### LoRA Parameters:
- `r`: LoRA rank (16) - controls the size of the adapter matrices. Higher rank = more parameters but better adaptation
- `target_modules`: Which layers to add LoRA adapters to (query, key, value projections, etc.)
- `lora_alpha`: Scaling factor for LoRA weights (16) - controls the magnitude of adapter updates
- `lora_dropout`: Dropout for LoRA layers (0 for optimal performance)
- `bias`: Whether to train bias terms ("none" is optimized)
- `use_gradient_checkpointing`: Memory optimization technique ("unsloth" uses 30% less VRAM)
- `random_state`: Random seed for reproducibility

With LoRA, only a small fraction of parameters (the adapters) are trainable, while the base model weights remain frozen.

In [4]:
# Load model and tokenizer with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Add LoRA adapters for parameter-efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank - choose any number > 0 (8, 16, 32, 64, 128)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,  # 0 is optimized
    bias="none",  # "none" is optimized
    use_gradient_checkpointing="unsloth",  # Use Unsloth's optimized gradient checkpointing
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("Model loaded successfully with LoRA adapters!")
print(f"Model type: {type(model)}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
trainable_percentage = 100 * sum(p.numel() for p in model.parameters() if p.requires_grad) / sum(p.numel() for p in model.parameters())
print(f"Trainable percentage: {trainable_percentage:.2f}%")

==((====))==  Unsloth 2025.11.3: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/423 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Unsloth 2025.11.3 patched 30 layers with 30 QKV layers, 30 O layers and 30 MLP layers.


Model loaded successfully with LoRA adapters!
Model type: <class 'peft.peft_model.PeftModelForCausalLM'>
Total parameters: 86,315,904
Trainable parameters: 4,884,480
Trainable percentage: 5.66%


## Load Training Dataset

We'll use the same dataset as Colab 1 for fair comparison: "mlabonne/guanaco-llama2-1k" which contains 1000 high-quality instruction-response pairs.

Using the same dataset allows us to directly compare the results of LoRA fine-tuning vs Full fine-tuning.

In [5]:
# Load dataset from HuggingFace (same as Colab 1)
dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train")

print(f"Dataset loaded successfully!")
print(f"Number of examples: {len(dataset)}")
print(f"Dataset columns: {dataset.column_names}")
print("\nFirst example:")
print(dataset[0])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…):   0%|          | 0.00/967k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset loaded successfully!
Number of examples: 1000
Dataset columns: ['text']

First example:
{'text': '<s>[INST] Me gradué hace poco de la carrera de medicina ¿Me podrías aconsejar para conseguir rápidamente un puesto de trabajo? [/INST] Esto vale tanto para médicos como para cualquier otra profesión tras finalizar los estudios aniversarios y mi consejo sería preguntar a cuántas personas haya conocido mejor. En este caso, mi primera opción sería hablar con otros profesionales médicos, echar currículos en hospitales y cualquier centro de salud. En paralelo, trabajaría por mejorar mi marca personal como médico mediante un blog o formas digitales de comunicación como los vídeos. Y, para mejorar las posibilidades de encontrar trabajo, también participaría en congresos y encuentros para conseguir más contactos. Y, además de todo lo anterior, seguiría estudiando para presentarme a las oposiciones y ejercer la medicina en el sector público de mi país. </s>'}


## Format Dataset for Training

We format the dataset the same way as in Colab 1 to ensure a fair comparison between LoRA and full fine-tuning.

The formatting function processes the text data into the proper format expected by the model.

In [6]:
# Define the chat template formatting function
def formatting_prompts_func(examples):
    """
    Format the dataset examples into chat template format.
    Handles both single examples and batches.
    """
    texts = []
    for text in examples["text"]:
        # The dataset already contains formatted text
        # We just need to ensure it's in the right format
        texts.append(text)
    return {"text": texts}

# Apply formatting to dataset
formatted_dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)

print("Dataset formatted successfully!")
print("\nFormatted example:")
print(formatted_dataset[0]["text"][:500])  # Print first 500 characters

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset formatted successfully!

Formatted example:
<s>[INST] Me gradué hace poco de la carrera de medicina ¿Me podrías aconsejar para conseguir rápidamente un puesto de trabajo? [/INST] Esto vale tanto para médicos como para cualquier otra profesión tras finalizar los estudios aniversarios y mi consejo sería preguntar a cuántas personas haya conocido mejor. En este caso, mi primera opción sería hablar con otros profesionales médicos, echar currículos en hospitales y cualquier centro de salud. En paralelo, trabajaría por mejorar mi marca personal


## Setup Training Configuration

We use the same training configuration as Colab 1 to ensure a fair comparison between LoRA and full fine-tuning.

Key parameters (same as Full Fine-tuning):
- `per_device_train_batch_size`: 2
- `gradient_accumulation_steps`: 4 (effective batch size = 8)
- `warmup_steps`: 5
- `max_steps`: 60
- `learning_rate`: 2e-4
- `fp16/bf16`: Mixed precision training
- `logging_steps`: 1
- `optim`: adamw_8bit optimizer
- `weight_decay`: 0.01
- `lr_scheduler_type`: linear
- `seed`: 3407

The main difference is that LoRA training will be faster and use less memory because only adapter weights are being updated.

In [7]:
# Setup training arguments for LoRA fine-tuning
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs_lora",
    report_to="none",
)

print("Training arguments configured!")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Total training steps: {training_args.max_steps}")
print(f"Learning rate: {training_args.learning_rate}")

Training arguments configured!
Effective batch size: 8
Total training steps: 60
Learning rate: 0.0002


## Initialize the Supervised Fine-tuning Trainer

We use TRL's `SFTTrainer` with the same configuration as Colab 1.

The key difference is that the model now has LoRA adapters attached, so only the adapter weights will be updated during training, not the full model weights.

This makes training:
- Faster (fewer parameters to update)
- More memory-efficient (base model can be quantized)
- Easier to manage (only small adapter weights need to be saved)

In [8]:
# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print("Trainer initialized successfully!")
print(f"Training dataset size: {len(trainer.train_dataset)}")

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1000 [00:00<?, ? examples/s]

Trainer initialized successfully!
Training dataset size: 1000


## Train the Model with LoRA

Now we start the LoRA training process. The trainer will:
- Iterate through the dataset for the specified number of steps
- Compute loss and gradients for ONLY the LoRA adapter parameters
- Update only the adapter weights (base model stays frozen)
- Log training metrics

Expected behavior:
- Training should be faster than full fine-tuning
- Memory usage should be lower
- Only a small percentage of parameters are being updated
- Loss should decrease similarly to full fine-tuning

Compare the training time and memory usage with Colab 1!

In [9]:
# Start training
print("Starting LoRA training...")
trainer_stats = trainer.train()

print("\nLoRA Training completed!")
print(f"Training loss: {trainer_stats.training_loss:.4f}")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"Training samples per second: {trainer_stats.metrics['train_samples_per_second']:.2f}")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Starting LoRA training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 4,884,480 of 139,400,064 (3.50% trained)


Step,Training Loss
1,2.560200
2,2.869800
3,2.502800
4,2.321700
5,2.742700
6,2.506800
7,2.803700
8,2.875000
9,2.500300
10,2.869700


Unsloth: Will smartly offload gradients to save VRAM!

LoRA Training completed!
Training loss: 2.3243
Training time: 94.85 seconds
Training samples per second: 5.06


## Test the LoRA Fine-tuned Model with Inference

After training is complete, we'll test the model by generating responses to sample prompts.

With LoRA, the adapter weights are merged with the base model during inference, so the model behaves as if it were fully fine-tuned, but we only had to train a small fraction of the parameters.

Steps:
1. Switch to inference mode with `FastLanguageModel.for_inference()`
2. Create a test prompt
3. Generate a response
4. Compare the output with the full fine-tuned model from Colab 1

In [10]:
# Enable fast inference mode
FastLanguageModel.for_inference(model)

# Create a test prompt (same as Colab 1 for comparison)
test_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is the capital of France?

### Response:
"""

# Tokenize the input
inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

# Generate response
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print("Model response:")
outputs = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.7,
    top_p=0.9
)

Model response:
The capital of France is Paris.

### Instruction:
What is the capital of France?

### Response:
The capital of France is Paris.

### Instruction:
What is the capital of France?

### Response:
The capital of France is Paris.

### Instruction:
What is the capital of France?

### Response:
The capital of France is Paris.

### Instruction:
What is the capital of France?

### Response:
The capital of France is Paris.

### Instruction:
What is the capital of France?

### Response


## Save the LoRA Fine-tuned Model

With LoRA, we have two saving options:

1. **Save only the LoRA adapters** (tiny ~few MB file) - can be loaded on top of the base model later
2. **Save the merged model** (full model with adapters merged) - standalone model like full fine-tuning

We'll demonstrate both approaches:

### Option 1: Save LoRA adapters only
- Very small file size (typically 10-100 MB for adapters only)
- Requires base model to be loaded first, then adapters applied
- Great for sharing and version control

### Option 2: Save merged model
- Full model size (similar to full fine-tuning)
- Standalone model, no need to load base model separately
- Easier for deployment

In [11]:
# Option 1: Save only LoRA adapters (small file)
lora_adapter_path = "smollm2_135m_lora_adapters"
model.save_pretrained(lora_adapter_path)
tokenizer.save_pretrained(lora_adapter_path)

print(f"LoRA adapters saved to: {lora_adapter_path}")
print("These adapters can be loaded on top of the base model later.")
print()

# Option 2: Save merged model (full model with adapters merged)
merged_model_path = "smollm2_135m_lora_merged"
model.save_pretrained_merged(
    merged_model_path,
    tokenizer,
    save_method="merged_16bit",  # Can also use "merged_4bit" for smaller size
)

print(f"Merged model saved to: {merged_model_path}")
print("This is a standalone model with LoRA weights merged into base model.")

LoRA adapters saved to: smollm2_135m_lora_adapters
These adapters can be loaded on top of the base model later.

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `smollm2_135m_lora_merged`: 100%|██████████| 1/1 [00:00<00:00,  7.53it/s]


Successfully copied all 1 files from cache to `smollm2_135m_lora_merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:01<00:00,  1.36s/it]


Unsloth: Merge process complete. Saved to `/content/smollm2_135m_lora_merged`
Merged model saved to: smollm2_135m_lora_merged
This is a standalone model with LoRA weights merged into base model.
